# Load data

In [1]:
import yaml
from IPython.display import display
from src.data import load_data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.metrics import r2_score
import duckdb
from scipy.optimize import curve_fit
import ipywidgets as widgets
from IPython.display import display

LOAD_FROM_DUCKDB = False

# with open("config.yaml", "r") as f:
#     cfg = yaml.safe_load(f)
 
# df, tmp_sensors, sensors = load_data(cfg['data'])
# df = df.sort_values(by="time").reset_index(drop=True)
# print(df1.head())

In [2]:
# load data from duckdb
if LOAD_FROM_DUCKDB:
    conn = duckdb.connect(r"C:\Users\matte\Desktop\NPlus\Git\timeseries_etl\data\P004_Adige_Est.duckdb")
    df = conn.execute("select * from main_staging.all_static").df()
else:
    df = pd.read_csv("C:\\Users\\matte\\Desktop\\NPlus\\01.Commesse\\Soglie_A4\\data\\_old_raw_df\\raw_df.csv")

In [3]:

# df["time"] = pd.to_datetime(df["time"])
tmp_sensors = [s for s in df.columns if s.endswith('_t')]
sensors = [s for s in df.columns if not s.endswith('_t') and s != 'time']
display(df.head(5))
display(df.tail(5))

,time,2301687F_t,2300989F_e,03068078_x,03068078_y,03068079_x,03068079_y,0306807E_x,0306807E_y,03068097_x,03068097_y,030680CB_x,030680CB_y,03068179_x,03068179_y,03119005_x,03119005_y,2302677F_s
0,2025-02-13 00:00:00,10.767551,18.776353,0.011500,0.360756,0.358492,-0.743155,-0.066914,-1.624124,-0.329683,0.019950,1.548134,0.951786,1.430788,0.255422,3.812496,0.657514,73.056324
1,2025-02-13 00:15:00,10.885479,18.791242,0.011699,0.360499,0.358572,-0.743464,-0.067110,-1.623923,-0.329664,0.020176,1.547889,0.951770,1.430509,0.255315,3.812926,0.657542,73.058129
2,2025-02-13 00:30:00,10.970815,18.790111,0.011700,0.360150,0.358462,-0.743236,-0.067139,-1.624095,-0.329709,0.020136,1.548121,0.951455,1.430927,0.255043,3.812671,0.657244,73.072869
3,2025-02-13 00:45:00,10.868391,18.821473,0.011672,0.360422,0.358461,-0.743182,-0.066961,-1.624061,-0.329818,0.019804,1.548256,0.951603,1.430956,0.255331,3.812783,0.657254,73.070953
4,2025-02-13 01:00:00,10.828333,18.833813,0.011617,0.360367,0.358875,-0.743095,-0.067067,-1.624090,-0.329622,0.020064,1.548169,0.951565,1.430628,0.255186,3.812555,0.657164,73.079368


,time,2301687F_t,2300989F_e,03068078_x,03068078_y,03068079_x,03068079_y,0306807E_x,0306807E_y,03068097_x,03068097_y,030680CB_x,030680CB_y,03068179_x,03068179_y,03119005_x,03119005_y,2302677F_s
22729,2025-10-29 22:45:00,14.812807,17.394981,0.006837,0.379783,0.364113,-0.733743,-0.059143,-1.607948,-0.339685,0.037846,1.543285,0.969141,1.428986,0.275615,3.814699,0.676934,71.163235
22730,2025-10-29 23:00:00,14.640989,17.395217,0.006492,0.379981,0.364322,-0.733880,-0.058981,-1.607996,-0.339478,0.037777,1.543224,0.969458,1.428964,0.275887,3.814645,0.676759,71.167028
22731,2025-10-29 23:15:00,14.765633,17.375382,0.006405,0.379812,0.364440,-0.734136,-0.058811,-1.608134,-0.339274,0.037654,1.543244,0.969117,1.428843,0.275774,3.814772,0.677071,71.180375
22732,2025-10-29 23:30:00,14.712676,17.391131,0.006549,0.379794,0.364502,-0.733951,-0.058594,-1.607724,-0.339333,0.037996,1.542804,0.969289,1.428696,0.275672,3.814555,0.676859,71.182296
22733,2025-10-29 23:45:00,14.592377,17.396453,0.006633,0.379840,0.364407,-0.733857,-0.058966,-1.607795,-0.339377,0.037702,1.542908,0.969581,1.428726,0.275874,3.814384,0.676979,71.184754


# Training

In [6]:
# assicura datetime
df["time"] = pd.to_datetime(df["time"])

# -----------------------------
# 1. FILTRO: UNA MISURA ALLE 5
# -----------------------------

target_hour = 4
tol = pd.Timedelta("1h")

rows = []

for d, g in df.groupby(df["time"].dt.date):

    target_time = pd.Timestamp(d) + pd.Timedelta(hours=target_hour)

    idx = (g["time"] - target_time).abs().idxmin()

    if abs(g.loc[idx, "time"] - target_time) <= tol:
        rows.append(idx)

df_filt = df.loc[rows].sort_values("time").reset_index(drop=True)


# -----------------------------
# 2. AZZERAMENTO
# -----------------------------

df_zero = df_filt.copy()

for col in sensors:
    df_zero[col] = df_zero[col] - df_zero[col].iloc[0]


def analyze_sensor(sensor_chosen):

    # -----------------------------
    # TRAIN / TEST SPLIT
    # -----------------------------
    split_idx = int(len(df_zero)*1.0)

    df_train = df_zero.iloc[:split_idx]
    df_test = df_zero.iloc[split_idx:]

    # -----------------------------
    # MODELLO TERMICO
    # -----------------------------
    T = df_train[tmp_sensors[0]].values
    eps = df_train[sensor_chosen].values

    model = lambda T, alpha, eps0: alpha*T + eps0

    A = np.vstack([T, np.ones(len(T))]).T
    alpha, epsilon0 = np.linalg.lstsq(A, eps, rcond=None)[0]
    param_opt, cov_opt = curve_fit(model, T, eps)
    print(param_opt)
    print(cov_opt)

    eps_pred = epsilon0 + alpha*T

    r2 = r2_score(eps, eps_pred)

    # -----------------------------
    # TERMOCOMPENSAZIONE
    # -----------------------------
    df_tc = df_zero.copy()
    df_tc[sensor_chosen + "_tc"] = df_zero[sensor_chosen] - alpha * df_zero[tmp_sensors[0]]

    # -----------------------------
    # FIGURA
    # -----------------------------
    fig = plt.figure(figsize=(16,8))

    gs = fig.add_gridspec(2, 2, width_ratios=[3,1])

    ax1 = fig.add_subplot(gs[0,:])   # serie temporale
    ax2 = fig.add_subplot(gs[1,0])   # model fit
    ax3 = fig.add_subplot(gs[1,1])   # scatter

    colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

    # -----------------------------
    # 1. SERIE TEMPORALE
    # -----------------------------
    ax1.plot(df_zero["time"], df_zero[sensor_chosen],
             color=colors[0], label=sensor_chosen)

    ax1.set_ylabel("displacement")

    ax1b = ax1.twinx()
    ax1b.plot(df_zero["time"], df_zero[tmp_sensors[0]],
              color=colors[1], label=tmp_sensors[0])

    ax1b.set_ylabel("temperature")

    ax1.set_title(f"Serie filtrata - {sensor_chosen}")

    ax1.xaxis.set_major_formatter(mdates.DateFormatter("%y-%m"))
    ax1.xaxis.set_major_locator(mdates.MonthLocator())

    ax1.grid(True)

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax1b.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2)

    # -----------------------------
    # 2. MODEL FIT
    # -----------------------------
    ax2.plot(df_train["time"], eps, label="original")
    ax2.plot(df_train["time"], eps_pred, label="predicted")

    ax2.set_title(f"Model fit (train)  R²={r2:.3f}")
    ax2.set_xlabel("time")
    ax2.set_ylabel("displacement")

    ax2.xaxis.set_major_formatter(mdates.DateFormatter("%y-%m"))
    ax2.xaxis.set_major_locator(mdates.MonthLocator())

    ax2.grid(True)
    ax2.legend()

    # -----------------------------
    # 3. SCATTER
    # -----------------------------
    ax3.scatter(df_train[tmp_sensors[0]], df_train[sensor_chosen], s=10)

    T_line = np.linspace(T.min(), T.max(), 100)
    eps_line = epsilon0 + alpha*T_line

    ax3.plot(T_line, eps_line, lw=1)

    ax3.set_xlabel("Temperature")
    ax3.set_ylabel("Displacement")
    ax3.set_title("Training Scatter")

    ax3.grid(True)

    plt.tight_layout()
    plt.show()

    print("alpha =", alpha)
    print("epsilon0 =", epsilon0)
    print("R² =", r2)


widgets.interact(
    analyze_sensor,
    sensor_chosen=widgets.Dropdown(
        options=sensors,
        value=sensors[-1],
        description="Sensor"
    )
)

interactive(children=(Dropdown(description='Sensor', index=15, options=('2300989F_e', '03068078_x', '03068078_…

<function __main__.analyze_sensor(sensor_chosen)>